# Découverte Sentinel-2 autour de Libreville

**Gabon GeoAI Lab - Observer · Comprendre · Prédire · Simuler**

Objectif : rechercher des produits et comparer leurs métadonnées. Aucun téléchargement d’image, aucune authentification, aucune clé ni écriture de données.

Dans VS Code, installer les extensions Python et Jupyter, puis choisir le noyau Python de `.venv` (3.10.11 ou 3.12). Depuis la racine du dépôt :

```bash
python -m pip install -r requirements-dev.txt
```

Exécuter les cellules dans l’ordre. Une connexion Internet est nécessaire.


## 1. Comprendre le catalogue

STAC (SpatioTemporal Asset Catalog) décrit des données associées à un lieu et à une date. Un catalogue contient des **collections** ; chaque produit est un **Item**. Les **assets** désignent les fichiers associés, que nous ne consultons ni ne téléchargeons ici.

Sentinel-2 **Level-2A (L2A)** fournit notamment une réflectance de surface après correction atmosphérique. Un produit contient plusieurs bandes et résolutions ; ce notebook n’effectue aucun traitement raster.

Sources officielles : [documentation STAC CDSE](https://documentation.dataspace.copernicus.eu/APIs/STAC.html), [catalogue public](https://stac.dataspace.copernicus.eu/v1/).


In [ ]:
from datetime import date
from math import isfinite
from urllib.parse import urljoin, urlparse

import requests

STAC_URL = "https://stac.dataspace.copernicus.eu/v1/"
COLLECTION_ID = "sentinel-2-l2a"
TIMEOUT_SECONDS = 40


## 2. Définir la zone pilote et la période

Une **bounding box** est un rectangle : `[longitude minimale, latitude minimale, longitude maximale, latitude maximale]`, soit ouest, sud, est, nord. Les coordonnées sont en degrés dans le référentiel géographique **WGS84**, avec la longitude en premier.

Notre rectangle va de 9,25° à 9,65° Est et de 0,20° à 0,65° Nord. Il couvre Libreville et ses environs, avec une partie côtière et maritime. C’est une zone expérimentale, pas une limite administrative.

Les produits recherchés **intersectent** ce rectangle ; cela ne garantit pas qu’un seul produit le couvre entièrement. Modifier les paramètres ci-dessous, puis réexécuter les cellules suivantes.

Le seuil nuageux sera appliqué localement aux métadonnées. La période est exprimée en UTC ; la date de fin comprend toute la journée.


In [ ]:
LIBREVILLE_BBOX = [9.25, 0.20, 9.65, 0.65]
START_DATE = "2025-01-01"
END_DATE = "2025-01-31"
MAX_CLOUD_COVER = 40.0  # Pourcentage entre 0 et 100.
TOP_N = 10
PAGE_SIZE = 100
MAX_PAGES = 10  # Limite pédagogique pour éviter une recherche illimitée.


In [ ]:
west, south, east, north = LIBREVILLE_BBOX
if not (-180 <= west < east <= 180 and -90 <= south < north <= 90):
    raise ValueError("Bounding box invalide : vérifier longitude et latitude.")
if date.fromisoformat(START_DATE) > date.fromisoformat(END_DATE):
    raise ValueError("La date de début doit précéder la date de fin.")
if not 0 <= MAX_CLOUD_COVER <= 100:
    raise ValueError("Le seuil nuageux doit être compris entre 0 et 100.")
if TOP_N < 1 or PAGE_SIZE < 1 or MAX_PAGES < 1:
    raise ValueError("Les limites doivent être positives.")

search_period = f"{START_DATE}T00:00:00Z/{END_DATE}T23:59:59.999Z"
print("Zone WGS84 :", LIBREVILLE_BBOX)
print("Période UTC :", search_period)


## 3. Vérifier la collection publique

Une requête HTTP GET lit un document JSON. `raise_for_status()` rend explicites les erreurs du serveur ; le délai limite l’attente. Aucune information d’identification n’est fournie. En cas d’échec réseau, corriger la connexion puis relancer la cellule.


In [ ]:
# Utilise le magasin de certificats du système d'exploitation.
# Utile notamment sur les postes d'entreprise utilisant leurs propres
# autorités de certification pour les connexions HTTPS.
import truststore

truststore.inject_into_ssl()

In [ ]:
response = requests.get(
    f"{STAC_URL}collections/{COLLECTION_ID}", timeout=TIMEOUT_SECONDS
)
response.raise_for_status()
collection = response.json()
if collection.get("id") != COLLECTION_ID:
    raise ValueError("La collection attendue n’a pas été retournée.")
print("Collection :", collection["id"])
print("Titre :", collection.get("title", "Non renseigné"))


## 4. Rechercher les métadonnées

`collections`, `bbox` et `datetime` limitent la recherche. `limit` fixe la taille demandée d’une page. Lorsqu’un lien STAC `next` est présent, nous suivons les pages suivantes jusqu’à la limite choisie.

Seules les métadonnées JSON du catalogue sont lues. Aucun lien d’asset n’est ouvert. Le tri final porte sur les résultats récupérés ; une recherche interrompue ou un catalogue partiel ne donne pas un classement exhaustif.


In [ ]:
search_params = {
    "collections": COLLECTION_ID,
    "bbox": ",".join(str(value) for value in LIBREVILLE_BBOX),
    "datetime": search_period,
    "limit": PAGE_SIZE,
}
items_by_id = {}
next_url = f"{STAC_URL}search"
params = search_params
search_truncated = False

for page_number in range(1, MAX_PAGES + 1):
    response = requests.get(next_url, params=params, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    page = response.json()
    if page.get("type") != "FeatureCollection":
        raise ValueError("La réponse n’est pas une collection de résultats STAC.")
    features = page.get("features", [])
    for item in features:
        items_by_id[item["id"]] = item
    print(f"Page {page_number} : {len(features)} produit(s)")

    next_link = next(
        (link for link in page.get("links", []) if link.get("rel") == "next"), None
    )
    if next_link is None:
        if len(features) >= PAGE_SIZE:
            print("Attention : page pleine sans lien suivant ; exhaustivité à vérifier.")
        break
    if page_number == MAX_PAGES:
        search_truncated = True
        break
    next_url = urljoin(response.url, next_link["href"])
    # Rester sur le catalogue public et sur une pagination GET.
    if (urlparse(next_url).scheme != "https"
            or urlparse(next_url).netloc != urlparse(STAC_URL).netloc
            or next_link.get("method", "GET").upper() != "GET"):
        raise ValueError("Lien de pagination non pris en charge : examiner les métadonnées.")
    params = None  # Le lien suivant porte ses propres paramètres.

items = list(items_by_id.values())
print(f"Total : {len(items)} produit(s) distinct(s)")
if search_truncated:
    print("Recherche limitée : réduire la période ou augmenter MAX_PAGES avant comparaison.")


## 5. Lire et afficher les résultats

`datetime` indique la date d’acquisition et `eo:cloud_cover` la couverture nuageuse en pourcentage : une valeur faible est généralement préférable pour observer la surface.

Cette valeur concerne **l’ensemble du produit**, pas uniquement notre rectangle. Elle ne suffit pas à déterminer la qualité locale : ombres, brume, couverture spatiale et masques de qualité devront être examinés dans une phase ultérieure.

Les valeurs absentes ou invalides sont affichées comme « Non renseignée » et exclues des candidates. Aucune valeur manquante n’est assimilée à 0 %.


In [ ]:
def read_cloud_cover(value):
    """Retourne un pourcentage valide ou None si la métadonnée manque."""
    if value is None or isinstance(value, bool):
        return None
    try:
        cloud_cover = float(value)
    except (TypeError, ValueError):
        return None
    return cloud_cover if isfinite(cloud_cover) and 0 <= cloud_cover <= 100 else None


records = []
for item in items:
    properties = item.get("properties", {})
    records.append({
        "id": item["id"],
        "acquired_at": properties.get("datetime"),
        "cloud_cover": read_cloud_cover(properties.get("eo:cloud_cover")),
    })


In [ ]:
def display_records(rows):
    """Affiche un tableau texte sans dépendance supplémentaire."""
    if not rows:
        print("Aucun résultat pour ces paramètres.")
        return
    print(f"{'ID':<75} | {'Acquisition UTC':<27} | Nuages (%)")
    print("-" * 120)
    for row in rows:
        cloud = row["cloud_cover"]
        cloud_label = f"{cloud:.2f}" if cloud is not None else "Non renseignée"
        acquired_at = row["acquired_at"] or "Non renseignée"
        print(f"{row['id']:<75} | {acquired_at:<27} | {cloud_label}")


display_records(records)


## 6. Classer les meilleures candidates

Nous conservons les produits dont la couverture nuageuse est renseignée et inférieure ou égale au seuil. Le classement privilégie la couverture nuageuse croissante, puis la date croissante et l’ID pour départager les égalités.

Il s’agit d’un classement des **produits** : plusieurs produits peuvent appartenir à une même acquisition. Il ne sélectionne pas encore une mosaïque couvrant toute la zone.


In [ ]:
candidates = [
    row for row in records
    if row["cloud_cover"] is not None and row["cloud_cover"] <= MAX_CLOUD_COVER
]
ranked_candidates = sorted(
    candidates,
    key=lambda row: (row["cloud_cover"], row["acquired_at"] or "", row["id"]),
)
missing_cloud_count = sum(row["cloud_cover"] is None for row in records)
print(f"Seuil : {MAX_CLOUD_COVER:.1f} % ; candidates : {len(ranked_candidates)}")
print(f"Couverture nuageuse absente ou invalide : {missing_cloud_count}")
display_records(ranked_candidates[:TOP_N])


## Pour apprendre et vérifier

1. Réexécuter toutes les cellules avec les paramètres par défaut et lire les trois colonnes.
2. Réduire le seuil nuageux à 20 %, puis relancer le classement : le nombre de candidates ne doit pas augmenter.
3. Changer le mois et relancer toutes les cellules : comparer la disponibilité et les nuages.
4. Vérifier que les candidates respectent le seuil et que la couverture nuageuse est croissante.
5. Si aucun produit ne passe le seuil, élargir la période ou relever le seuil explicitement.

Avant de partager le notebook, effacer ses sorties. Aucun fichier d’image, donnée raster ou secret ne doit être ajouté au dépôt.


In [ ]:
best_candidate_id = ranked_candidates[0]["id"] if ranked_candidates else None

if best_candidate_id is None:
    print("Aucun candidat disponible.")
else:
    best_item = items_by_id[best_candidate_id]

    print("Produit sélectionné :", best_candidate_id)
    print("\nAssets disponibles :")

    for asset_name, asset in best_item.get("assets", {}).items():
        print(
            asset_name,
            "|",
            asset.get("type", "type non renseigné"),
            "|",
            asset.get("title", "titre non renseigné"),
        )

In [ ]:
if best_candidate_id is not None:
    b04_asset = best_item["assets"]["B04_10m"]

    print("Asset :", "B04_10m")
    print("Type :", b04_asset.get("type"))
    print("Titre :", b04_asset.get("title"))
    print("Adresse :", b04_asset.get("href"))

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd().parent
load_dotenv(project_root / ".env")

s3_access_key = os.getenv("CDSE_S3_ACCESS_KEY")
s3_secret_key = os.getenv("CDSE_S3_SECRET_KEY")

if not s3_access_key or not s3_secret_key:
    raise RuntimeError("Credentials S3 Copernicus introuvables.")

print("Credentials S3 chargés :", bool(s3_access_key and s3_secret_key))

In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="https://eodata.dataspace.copernicus.eu",
    aws_access_key_id=s3_access_key,
    aws_secret_access_key=s3_secret_key,
    region_name="default",
)

bucket_name = "eodata"

b04_s3_uri = b04_asset["href"]
b04_key = b04_s3_uri.removeprefix("s3://eodata/")

print("Bucket :", bucket_name)
print("Fichier :", b04_key.split("/")[-1])

In [ ]:
metadata = s3.head_object(
    Bucket=bucket_name,
    Key=b04_key,
)

size_mb = metadata["ContentLength"] / (1024 * 1024)

print("Accès S3 : OK")
print(f"Taille du fichier B04 : {size_mb:.1f} Mo")
print("Type :", metadata.get("ContentType", "Non renseigné"))

## 6. Lecture du raster Sentinel-2

Objectif : ouvrir la bande B04 avec Rasterio et comprendre sa structure géospatiale avant toute analyse.

In [ ]:
import rasterio

In [ ]:
import os

os.environ["AWS_S3_ENDPOINT"] = "eodata.dataspace.copernicus.eu"
os.environ["AWS_ACCESS_KEY_ID"] = s3_access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = s3_secret_key
os.environ["AWS_HTTPS"] = "YES"
os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"
os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"

In [ ]:
from pathlib import Path

raw_dir = Path("../data/raw/sentinel2")
raw_dir.mkdir(parents=True, exist_ok=True)

b04_local_path = raw_dir / b04_key.split("/")[-1]

print(b04_local_path)

In [ ]:
if not b04_local_path.exists():
    print("Téléchargement de B04...")
    
    s3.download_file(
        bucket_name,
        b04_key,
        str(b04_local_path),
    )
    
    print("Téléchargement terminé.")
else:
    print("B04 existe déjà localement.")

In [ ]:
with rasterio.open(b04_local_path) as src:
    print("Driver :", src.driver)
    print("Largeur :", src.width)
    print("Hauteur :", src.height)
    print("Nombre de bandes :", src.count)
    print("Type des pixels :", src.dtypes)
    print("CRS :", src.crs)
    print("Résolution :", src.res)
    print("Emprise :", src.bounds)

In [ ]:
with rasterio.open(b04_local_path) as src:
    b04_preview = src.read(
        1,
        out_shape=(1000, 1000),
    )

print("Dimensions de l'aperçu :", b04_preview.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
plt.imshow(b04_preview, cmap="gray")
plt.title("Sentinel-2 - Bande B04 (rouge)")
plt.axis("off")
plt.show()

In [ ]:
print("Minimum :", b04_preview.min())
print("Maximum :", b04_preview.max())
print("Moyenne :", b04_preview.mean())
print("Type :", b04_preview.dtype)

In [ ]:
for band in ["B02_10m", "B03_10m", "B04_10m"]:
    asset = best_item["assets"].get(band)
    print(band, "→", asset["href"] if asset else "Introuvable")

In [ ]:
def download_band(band_name):
    asset = best_item["assets"].get(band_name)

    if asset is None:
        raise ValueError(f"Bande introuvable : {band_name}")

    s3_uri = asset["href"]
    s3_key = s3_uri.removeprefix("s3://eodata/")
    local_path = raw_dir / s3_key.split("/")[-1]

    if not local_path.exists():
        print(f"Téléchargement de {band_name}...")
        s3.download_file(
            bucket_name,
            s3_key,
            str(local_path),
        )
        print(f"{band_name} téléchargée.")
    else:
        print(f"{band_name} existe déjà.")

    return local_path

In [ ]:
b02_local_path = download_band("B02_10m")
b03_local_path = download_band("B03_10m")
b04_local_path = download_band("B04_10m")

In [ ]:
with rasterio.open(b02_local_path) as src:
    b02_preview = src.read(1, out_shape=(1000, 1000))

with rasterio.open(b03_local_path) as src:
    b03_preview = src.read(1, out_shape=(1000, 1000))

with rasterio.open(b04_local_path) as src:
    b04_preview = src.read(1, out_shape=(1000, 1000))

In [ ]:
print("B02 :", b02_preview.shape)
print("B03 :", b03_preview.shape)
print("B04 :", b04_preview.shape)

In [ ]:
import numpy as np

In [ ]:
rgb_preview = np.dstack((b04_preview, b03_preview, b02_preview))

In [ ]:
print(rgb_preview.shape)
print(rgb_preview[500, 300])

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(rgb_preview)
plt.title("Sentinel-2 RGB brut")
plt.axis("off")
plt.show()

In [ ]:
def stretch_band(band):
    p2, p98 = np.percentile(band, (2, 98))

    stretched = (band - p2) / (p98 - p2)

    return np.clip(stretched, 0, 1)

In [ ]:
r = stretch_band(b04_preview)
g = stretch_band(b03_preview)
b = stretch_band(b02_preview)

rgb_stretched = np.dstack((r, g, b))

In [ ]:
print("Shape :", rgb_stretched.shape)
print("Minimum :", rgb_stretched.min())
print("Maximum :", rgb_stretched.max())

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(rgb_stretched)
plt.title("Sentinel-2 - Couleurs naturelles")
plt.axis("off")
plt.show()

In [ ]:
b08_asset = best_item["assets"].get("B08_10m")

print(b08_asset["href"] if b08_asset else "B08_10m introuvable")

In [ ]:
b08_local_path = download_band("B08_10m")

In [ ]:
with rasterio.open(b08_local_path) as src:
    b08_preview = src.read(
        1,
        out_shape=(1000, 1000),
    )

print("B08 :", b08_preview.shape)
print("Minimum :", b08_preview.min())
print("Maximum :", b08_preview.max())
print("Type :", b08_preview.dtype)

In [ ]:
from pathlib import Path
from src.data.radiometric_calibration import read_sentinel2_calibration
from src.data.cloud_mask import create_scl_mask
from src.geo.vegetation_indices import calculate_ndvi

# XML du même produit que les bandes ; aucune calibration implicite.
calibration_xml = Path(b04_local_path).parent / "MTD_MSIL2A.xml"
calibration = read_sentinel2_calibration(calibration_xml)
scl_local_path = download_band("SCL_20m")
scl_policy = "usable_land"  # SCL 4 et 5 ; ombres 2 exclues.
valid_mask = create_scl_mask(scl_local_path, b04_local_path, policy=scl_policy)
ndvi_native, ndvi_profile = calculate_ndvi(
    red_path=b04_local_path,
    nir_path=b08_local_path,
    valid_mask=valid_mask,
    red_offset=calibration["offsets"]["B04"],
    nir_offset=calibration["offsets"]["B08"],
    quantification_value=calibration["quantification_value"],
)
# Aperçu sous-échantillonné après calcul sur la grille native.
ndvi = ndvi_native[::10, ::10]

In [ ]:
print("Shape :", ndvi.shape)
print("Minimum :", np.nanmin(ndvi_native))
print("Maximum :", np.nanmax(ndvi_native))
print("Moyenne :", np.nanmean(ndvi_native))

In [ ]:
plt.figure(figsize=(8, 8))

plt.imshow(
    ndvi,
    cmap="RdYlGn",
    vmin=-1,
    vmax=1
)

plt.colorbar(label="NDVI")
plt.title("GEONAF - NDVI Sentinel-2")
plt.axis("off")
plt.show()

In [ ]:
for asset_name in best_item["assets"]:
    if "SCL" in asset_name.upper():
        print(asset_name)

In [ ]:
# SCL déjà téléchargée pour le calcul canonique ci-dessus.
print("Politique SCL :", scl_policy)

In [ ]:
with rasterio.open(scl_local_path) as src:
    print("Dimensions :", src.shape)
    print("Résolution :", src.res)
    print("CRS :", src.crs)
    print("Type :", src.dtypes)

In [ ]:
from rasterio.enums import Resampling

In [ ]:
with rasterio.open(scl_local_path) as src:
    scl_preview = src.read(
        1,
        out_shape=(1000, 1000),
        resampling=Resampling.nearest,
    )

print("Shape :", scl_preview.shape)
print("Type :", scl_preview.dtype)
print("Classes présentes :", np.unique(scl_preview))

In [ ]:
with rasterio.open(scl_local_path) as src:
    scl_native = src.read(1)

print("Classes natives :", np.unique(scl_native))

In [ ]:
cloud_classes = [3, 8, 9, 10]

cloud_mask_native = np.isin(
    scl_native,
    cloud_classes
)

In [ ]:
print("Shape :", cloud_mask_native.shape)
print("Type :", cloud_mask_native.dtype)

cloud_percentage = cloud_mask_native.mean() * 100
print("Pixels masqués :", round(cloud_percentage, 2), "%")

In [ ]:
# Masque analytique issu de la grille native, comme le NDVI.
cloud_mask = ~valid_mask[::10, ::10]

In [ ]:
print("NDVI :", ndvi.shape)
print("Masque analytique :", cloud_mask.shape)

In [ ]:
from rasterio.warp import reproject

In [ ]:
with rasterio.open(b04_local_path) as red_src:
    red_transform = red_src.transform
    red_crs = red_src.crs
    red_width = red_src.width
    red_height = red_src.height

with rasterio.open(scl_local_path) as scl_src:
    scl_aligned = np.empty(
        (red_height, red_width),
        dtype=np.uint8
    )

    reproject(
        source=rasterio.band(scl_src, 1),
        destination=scl_aligned,
        src_transform=scl_src.transform,
        src_crs=scl_src.crs,
        dst_transform=red_transform,
        dst_crs=red_crs,
        resampling=Resampling.nearest,
    )

In [ ]:
print("Shape :", scl_aligned.shape)
print("Classes :", np.unique(scl_aligned))

In [ ]:
# Le calcul canonique vérifie CRS, transformée et dimensions B04/B08.
print("NDVI natif :", ndvi_native.shape)

In [ ]:
cloud_mask = ~valid_mask
print("Pixels exclus par SCL :", round(cloud_mask.mean() * 100, 2), "%")

In [ ]:
# NDVI natif déjà calibré BOA et masqué par le calcul canonique.

In [ ]:
ndvi_clean = ndvi_native.copy()  # Aucun second calcul ni second masquage.

In [ ]:
print("NDVI min :", np.nanmin(ndvi_clean))
print("NDVI max :", np.nanmax(ndvi_clean))
print("NDVI moyen :", np.nanmean(ndvi_clean))
print("Pixels exclus :", round(np.isnan(ndvi_clean).mean() * 100, 2), "%")

In [ ]:
step = 10
ndvi_display = ndvi_clean[::step, ::step]

print("Affichage :", ndvi_display.shape)

In [ ]:
plt.figure(figsize=(8, 8))

plt.imshow(
    ndvi_display,
    cmap="RdYlGn",
    vmin=-1,
    vmax=1
)

plt.colorbar(label="NDVI")
plt.title("GEONAF - NDVI avec masque nuages")
plt.axis("off")
plt.show()